In [ ]:
import pandas as pd
import os

input_file  = r"C:\Users\lenovo\Desktop\ARIA.csv"
output_file = r"C:\Users\lenovo\Desktop\ARIA_RAG_READY.csv"

print("Chargement du fichier ARIA (53k lignes)...")
df = pd.read_csv(input_file, low_memory=False, encoding="utf-8", sep=",", on_bad_lines='skip')

print(f"Lignes totales au départ : {len(df):,}")

# 1. Garde seulement les colonnes vraiment utiles pour un RAG
useful_cols = ['Titre', 'Date', 'Département', 'Commune',', "Type d'accident", 'Matière', 'Equipements', 'Classe de danger CLP', 'Causes profondes', 'Conséquences', 'Contenu']
df = df[useful_cols].copy()

# 2. Filtre ultra-efficace : on garde seulement les accidents avec un récit détaillé
df = df.dropna(subset=['Contenu'])                                     # Contenu obligatoire
df = df[df['Contenu'].str.len() >= 150]                                # au moins ~25 mots
df = df[df['Contenu'].str.strip() != ""]                               # pas juste des espaces

# 3. Nettoyage supplémentaire
df.drop_duplicates(subset=['Numéro ARIA'], inplace=True)               # normalement déjà unique mais au cas où
df.drop_duplicates(subset=['Contenu'], inplace=True)                   # vire les copier-coller identiques

# 4. Option : tu peux garder seulement les plus graves / récents si tu veux encore réduire
# df = df[df['Date'].str[:4].astype(int) >= 2015]     # seulement 2015+
# df = df[df['Conséquences'].str.contains('mort|blessé grave', case=False, na=False)]

# Reset index pour que ce soit propre
df.reset_index(drop=True, inplace=True)

# Sauvegarde
df.to_csv(output_file, index=False, encoding="utf-8")
size_mb = os.path.getsize(output_file) / (1024*1024)

print("\n" + "="*70)
print("RAG READY - Fichier réduit et propre !")
print(f"Lignes gardées          : {len(df):,}  (sur 53 958)")
print(f"Fichier final           : {output_file}")
print(f"Taille finale           : {size_mb:.1f} Mo")
print("\nExemple des 3 meilleurs récits (les plus longs) :")
print(df.nlargest(3, df['Contenu'].str.len())[['Titre', 'Date', 'Contenu']].to_string(index=False))
print("="*70)

SyntaxError: unterminated string literal (detected at line 13) (1926876733.py, line 13)